# NB01 — Inspección del modelo AsymMirai

**TFM · Máster Universitario en Inteligencia Artificial · VIU 2025-2026 · Víctor Rodríguez Rodríguez**

---

## Propósito
Cargar AsymMirai, verificar accesibilidad a sus componentes internos, identificar decisiones de diseño relevantes para la extracción (alignment, stretch, shapes), y confirmar que se puede reutilizar como extractor congelado de features.

**Inputs**: `AsymMirai/snapshots/trained_asymmirai.pt`.

---

## Qué se necesita verificar antes de extraer features

Antes de lanzar la extracción sobre miles de imágenes hay que responder cuatro preguntas cuyas respuestas condicionan directamente el código del pipeline:

1. **¿Cómo se accede al backbone?** El modelo puede venir envuelto en `nn.DataParallel` (paralelización multi-GPU heredada del entrenamiento original), lo que añade una capa intermedia `.module` que hay que desenrollar para acceder al `nn.Sequential` interno.
2. **¿Qué shape devuelve el backbone por vista?** Necesitamos conocerlo para dimensionar correctamente las cabezas clasificadoras y las estrategias de reducción espacial.
3. **¿Hay que voltear horizontalmente la mama derecha antes del cálculo bilateral?** Depende del atributo `alignment_space`: `'pixel'` exige flip antes del backbone, `'latent'` exige flip después, `None` no requiere ninguno.
4. **¿Están activos los `stretch params`?** Si `use_stretch=True`, existen vectores aprendidos `cc_stretch_params` y `mlo_stretch_params` de dimensión 512 que multiplican canal a canal los embeddings antes de calcular la diferencia bilateral. Replicarlos es imprescindible si queremos que la extracción en el Punto B refleje la operación real de AsymMirai.

In [1]:
import os, sys
import torch

BASE = os.environ.get('TFM_PROJECT_ROOT', os.path.abspath(os.path.join(os.getcwd(), '..')))
ASYMMIRAI = os.path.join(BASE, 'AsymMirai')
WEIGHTS = os.path.join(ASYMMIRAI, 'snapshots', 'trained_asymmirai.pt')

# El modelo está serializado con torch.save y referencia clases definidas dentro de AsymMirai. Se añaden esas rutas al sys.path para que PyTorch pueda reconstruir el objeto al cargarlo.
sys.path.insert(0, ASYMMIRAI)
sys.path.insert(0, os.path.join(ASYMMIRAI, 'asymmetry_model'))

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

Device: cuda


## 1. Cargar el modelo y congelar pesos

`weights_only=False` es necesario porque el checkpoint contiene el objeto entero (no solo el state_dict). `model.eval()` desactiva dropout y pone BatchNorm en modo inferencia. Y congelamos los gradientes porque no se entrena nada del backbone, solo se usa como extractor.

In [2]:
model = torch.load(WEIGHTS, map_location=DEVICE, weights_only=False)
model.eval()
for p in model.parameters():
    p.requires_grad = False

print(f'Tipo de modelo: {type(model).__name__}')
print(f'Parámetros totales: {sum(p.numel() for p in model.parameters()):,}')
print(f'En device: {next(model.parameters()).device}')

c:\Users\victo\miniconda3\envs\tfm\Lib\site-packages\torch\serialization.py:1782: SourceChangeWarning: source code of class 'mirai_localized_dif_head.LocalizedDifModel' has changed. you can retrieve the original source code by accessing the object's source attribute or set `torch.nn.Module.dump_patches = True` and use the patch tool to revert the changes.
  _check_container_source(*data)
c:\Users\victo\miniconda3\envs\tfm\Lib\site-packages\torch\serialization.py:1782: SourceChangeWarning: source code of class 'torch.nn.modules.container.Sequential' has changed. you can retrieve the original source code by accessing the object's source attribute or set `torch.nn.Module.dump_patches = True` and use the patch tool to revert the changes.
  _check_container_source(*data)


Tipo de modelo: LocalizedDifModel
Parámetros totales: 11,176,512
En device: cuda:0


## 2. Atributos relevantes del modelo

- **`alignment_space`**: indica cómo se alinean las mamas L y R para comparar. `None` significa que ambas se procesan tal cual (sin flip). `'pixel'` exigiría voltear R horizontalmente antes del backbone. `'latent'` exigiría voltear el embedding de R después del backbone.
- **`use_stretch`**: si su valor es `True`, hay parámetros `cc_stretch_params` y `mlo_stretch_params` que el modelo aprende y multiplica canal a canal sobre los embeddings antes de calcular `|L − R|`.
- **`train_backbone`**: indica si fue entrenable durante el entrenamiento original. En este caso no influye ya que lo congelamos.
- **`latent_h`, `latent_w`**: dimensiones de la rejilla latente sobre la que se computa la asimetría localizada.

In [3]:
atributos_relevantes = [
    'alignment_space', 'latent_h', 'latent_w',
    'flexible_asymmetry', 'use_stretch', 'use_stretch_matrix',
    'use_bn', 'train_backbone', 'topk_for_heatmap',
    'learned_asym_mean', 'learned_asym_std',
]
for a in atributos_relevantes:
    if hasattr(model, a):
        v = getattr(model, a)
        if isinstance(v, torch.Tensor):
            v = v.item() if v.numel() == 1 else v.shape
        print(f'{a} = {v}')
    else:
        print(f'{a} = no existe')

alignment_space = None
latent_h = 5
latent_w = 5
flexible_asymmetry = True
use_stretch = True
use_stretch_matrix = False
use_bn = False
train_backbone = False
topk_for_heatmap = None
learned_asym_mean = 40
learned_asym_std = 10


## 3. Estructura del backbone

El backbone es un `nn.Sequential` con 9 bloques: un `Downsampler` que reduce la resolución espacial, seguido de 8 `BasicBlock` (los bloques estándar de ResNet-18).

In [4]:
# Comprobar si el modelo se encuentra envuelto en DataParallel.
if isinstance(model.backbone, torch.nn.DataParallel):
    backbone = model.backbone.module
    print('Envuelto en DataParallel, usando .module')
else:
    backbone = model.backbone
    print('No DataParallel, usando directamente')

print(f'\nTipo interno: {type(backbone).__name__}')
print(f'Parámetros backbone: {sum(p.numel() for p in backbone.parameters()):,}')
print(f'\nCapas top-level:')
for i, (name, mod) in enumerate(backbone.named_children()):
    print(f'{name:20s}  {type(mod).__name__}')

No DataParallel, usando directamente

Tipo interno: Sequential
Parámetros backbone: 11,176,512

Capas top-level:
0                     Downsampler
1                     BasicBlock
2                     BasicBlock
3                     BasicBlock
4                     BasicBlock
5                     BasicBlock
6                     BasicBlock
7                     BasicBlock
8                     BasicBlock


## 4. Forward de prueba: ¿qué shape devuelve el backbone?

Pasamos un tensor del shape que el modelo espera (`1, 3, 1664, 2048`: batch, canales RGB, alto, ancho) y vemos qué sale. Lo que salga es la representación que extraeremos por vista en el Punto A.

In [5]:
# Tensor de prueba para inspeccionar la salida del backbone.
x_dummy = torch.randn(1, 3, 1664, 2048, device=DEVICE)
print(f'Entrada: {tuple(x_dummy.shape)}')

with torch.no_grad():
    out = model.backbone(x_dummy)

print(f'Salida del backbone (PUNTO A): {tuple(out.shape)}')
print(f'dtype: {out.dtype}')
print(f'rango: [{out.min().item():.3f}, {out.max().item():.3f}]')
print(f'media: {out.mean().item():.3f}')
print(f'memoria por estudio (4 vistas) si lo guardáramos crudo: {4*out.numel()*4 / 1e6:.1f} MB')

Entrada: (1, 3, 1664, 2048)
Salida del backbone (PUNTO A): (1, 512, 52, 64)
dtype: torch.float32
rango: [0.000, 7.517]
media: 0.211
memoria por estudio (4 vistas) si lo guardáramos crudo: 27.3 MB


## 5. Cálculo del Punto B (asimetría bilateral) — prueba con tensores aleatorios

Replicar el funcionamiento interno de AsymMirai, en este caso no es necesario aplicar flip como ya se observó en el análisis de atributos relevantes. Calcular `|L − R|` sobre los embeddings.

In [6]:
L = torch.randn(1, 3, 1664, 2048, device=DEVICE)
R = torch.randn(1, 3, 1664, 2048, device=DEVICE)

with torch.no_grad():
    L_emb = model.backbone(L)
    R_emb = model.backbone(R)
    
    # Comprobación extra sobre el flip.
    if getattr(model, 'alignment_space', None) == 'latent':
        print('alignment_space == "latent". Aplicar flip de R_emb en dim -1')
        R_emb = torch.flip(R_emb, dims=[-1])
    elif getattr(model, 'alignment_space', None) == 'pixel':
        print('alignment_space == "pixel". Aplicar flip a la imagen antes del backbone')
    else:
        print('alignment_space == None. No se debe aplicar flip, ambas vistas se procesan tal cual')
    
    asym_map = torch.abs(L_emb - R_emb)

print(f'\nPUNTO B (mapa de asimetría bilateral): {tuple(asym_map.shape)}')
print(f'memoria por estudio (2 pares CC + MLO) si lo guardáramos crudo: {2*asym_map.numel()*4 / 1e6:.1f} MB')
print(f'rango: [{asym_map.min().item():.3f}, {asym_map.max().item():.3f}]')

alignment_space == None. No se debe aplicar flip, ambas vistas se procesan tal cual

PUNTO B (mapa de asimetría bilateral): (1, 512, 52, 64)
memoria por estudio (2 pares CC + MLO) si lo guardáramos crudo: 13.6 MB
rango: [0.000, 3.507]


## 6. Resumen

Lo que necesitamos para la extracción está confirmado:

- `model.backbone()`. Input con shape `(B, 3, 1664, 2048)`, output embedding `(B, 512, 52, 64)`.
- `alignment_space=None`. No hay que voltear la mama derecha.
- `use_stretch=True`. Hay que aplicar `cc_stretch_params` y `mlo_stretch_params` (vectores de 512) antes de calcular la diferencia bilateral, multiplicando canal a canal.